In [1]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

assert groq_key, "Missing GROQ_API_KEY"
assert gemini_key, "Missing GEMINI_API_KEY — get one free at aistudio.google.com, no card needed"

groq_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print(" Groq key loaded")
print(" Gemini key loaded")

 Groq key loaded
 Gemini key loaded


In [2]:
!pip install -q google-genai
print(" google-genai installed")

 google-genai installed



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from google import genai

gemini_client = genai.Client(api_key=gemini_key)

for m in gemini_client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [4]:
import time

def call_gemini_with_retry(model, contents, config=None, max_retries=3):
    for attempt in range(max_retries):
        try:
            if config:
                return gemini_client.models.generate_content(model=model, contents=contents, config=config)
            else:
                return gemini_client.models.generate_content(model=model, contents=contents)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"Attempt {attempt+1} failed ({e.__class__.__name__}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

In [5]:
import time

def call_gemini_with_retry(model, contents, config=None, max_retries=3):
    for attempt in range(max_retries):
        try:
            if config:
                return gemini_client.models.generate_content(model=model, contents=contents, config=config)
            else:
                return gemini_client.models.generate_content(model=model, contents=contents)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt  # 1s, 2s, 4s
                print(f"Attempt {attempt+1} failed ({e.__class__.__name__}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

response = call_gemini_with_retry("gemini-flash-latest", "Reply with exactly: Gemini connection successful.")
print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Attempt 1 failed (ServerError), retrying in 1s...
Attempt 2 failed (ServerError), retrying in 2s...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
response = call_gemini_with_retry("gemini-3.6-flash", "Reply with exactly: Gemini connection successful.")
print(response.text)

Gemini connection successful.


In [ ]:
from google.genai import types

response = call_gemini_with_retry(
    "gemini-3.6-flash",
    "What are the latest developments in EV battery technology in 2026? Give 3 points with sources.",
    config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
)

print("--- Answer ---")
print(response.text)

print("\n--- Grounding sources ---")
if response.candidates[0].grounding_metadata:
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        print("-", chunk.web.title, "|", chunk.web.uri)

Attempt 1 failed (ClientError), retrying in 1s...
Attempt 2 failed (ClientError), retrying in 2s...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [6]:
from typing import TypedDict, Annotated, List, Dict, Optional
from operator import add

class ResearchState(TypedDict):
    query: str
    plan: List[str]
    needs_more_research: bool
    search_results: Annotated[List[Dict], add]
    extracted_facts: Annotated[List[Dict], add]
    fact_check_results: Annotated[List[Dict], add]
    summaries: Annotated[List[Dict], add]
    final_report: Optional[str]
    model_usage_log: Annotated[List[Dict], add]
    agent_log: Annotated[List[str], add]

print(" ResearchState refined")
print(ResearchState.__annotations__)

 ResearchState refined
{'query': <class 'str'>, 'plan': typing.List[str], 'needs_more_research': <class 'bool'>, 'search_results': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'extracted_facts': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'fact_check_results': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'summaries': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'final_report': typing.Optional[str], 'model_usage_log': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'agent_log': typing.Annotated[typing.List[str], <built-in function add>]}


In [7]:
def get_llm_for_role(role: str):
    """Central place that decides which model handles which agent role.
    Groq = fast reasoning/classification/orchestration.
    Gemini = search grounding, once their free tier stabilizes — using Tavily for now."""
    if role in ("planner", "fact_checker", "summarizer", "supervisor"):
        return "groq"
    elif role in ("researcher",):
        return "tavily"   # temporary — swap to "gemini" once their API is stable
    else:
        raise ValueError(f"Unknown role: {role}")

print(" Model router defined")

 Model router defined


In [ ]:
def planner_node(state: ResearchState) -> dict:
    print(" Planner running (Groq)...")
    prompt = f"""Break this research question into sub-questions: {state['query']}
Reply with each sub-question on its own line, nothing else."""
    response = groq_llm.invoke(prompt)
    sub_questions = [line.strip("- ").strip() for line in response.content.split("\n") if line.strip()]

    classify_prompt = f"""Is this research query specific enough to research directly, or is it too
vague/broad? Query: "{state['query']}"
Reply with exactly one word: SPECIFIC or VAGUE."""
    classification = groq_llm.invoke(classify_prompt).content.strip().upper()
    needs_more = "VAGUE" in classification

    return {
        "plan": sub_questions,
        "needs_more_research": needs_more,
        "model_usage_log": [{"agent": "planner", "model": "groq/openai-gpt-oss-120b"}],
        "agent_log": [f"planner_node: generated {len(sub_questions)} sub-questions, classified as {classification}"]
    }

def researcher_node(state: ResearchState) -> dict:
    print(" Researcher running (Tavily)...")
    from tavily import TavilyClient
    tavily = TavilyClient(api_key=tavily_key)

    results = []
    for sub_q in state["plan"][:3]:
        search = tavily.search(query=sub_q, max_results=3)
        sources = [{"title": r["title"], "url": r["url"]} for r in search["results"]]
        answer = " ".join(r.get("content", "")[:200] for r in search["results"])
        results.append({"sub_question": sub_q, "answer": answer, "sources": sources})

    return {
        "search_results": results,
        "model_usage_log": [{"agent": "researcher", "model": "tavily"}],
        "agent_log": [f"researcher_node: researched {len(results)} sub-questions via Tavily"]
    }

def clarify_node(state: ResearchState) -> dict:
    print(" Clarify running...")
    return {"agent_log": ["clarify_node: flagged query as needing clarification"]}

print(" Nodes defined")

 Nodes defined


In [9]:
def route_after_planning(state: ResearchState) -> str:
    if state["needs_more_research"]:
        print("Router: too vague → 'clarify'")
        return "clarify"
    else:
        print(" Router: plan solid → 'researcher'")
        return "researcher"

In [10]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(ResearchState)
graph.add_node("planner", planner_node)
graph.add_node("researcher", researcher_node)
graph.add_node("clarify", clarify_node)

graph.add_edge(START, "planner")
graph.add_conditional_edges("planner", route_after_planning, {"researcher": "researcher", "clarify": "clarify"})
graph.add_edge("researcher", END)
graph.add_edge("clarify", END)

app = graph.compile()
print(" Graph compiled")

 Graph compiled


In [11]:
result = app.invoke({
    "query": "What are the current safety concerns with solid-state EV batteries?",
    "plan": [], "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "final_report": None,
    "model_usage_log": [], "agent_log": []
})

print("--- Agent log ---")
for line in result["agent_log"]:
    print(line)

print("\n--- Model usage ---")
for m in result["model_usage_log"]:
    print(m)

print("\n--- Search results ---")
for r in result["search_results"]:
    print("\nQ:", r["sub_question"])
    print("A:", r["answer"][:200], "...")
    print("Sources:", r["sources"])

 Planner running (Groq)...
 Router: plan solid → 'researcher'
 Researcher running (Tavily)...
--- Agent log ---
planner_node: generated 10 sub-questions, classified as SPECIFIC
researcher_node: researched 3 sub-questions via Tavily

--- Model usage ---
{'agent': 'planner', 'model': 'groq/openai-gpt-oss-120b'}
{'agent': 'researcher', 'model': 'tavily'}

--- Search results ---

Q: What are the primary chemical degradation mechanisms in solid‑state EV batteries?
A: In all-solid-state batteries, the cathode and anode have a volume change during repeated charging and discharging, resulting in interfacial degradation such as side reaction and contact loss between a ...
Sources: [{'title': 'Investigation of degradation mechanism for all-solid-state batteries takes another step toward commercialization | Newswise', 'url': 'https://www.newswise.com/articles/investigation-of-degradation-mechanism-for-all-solid-state-batteries-takes-another-step-toward-commercialization'}, {'title': 'Battery Degr

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Tracing enabled:", os.getenv("LANGSMITH_TRACING"))
print("Project:", os.getenv("LANGSMITH_PROJECT"))